# Data Preprocessing of the Formula 1 Dataset

**Assignment:** Prepare the Formula 1 CSV files for upload to the college PostgreSQL database
**Source Data:** Formula 1 World Championship (1950–2024), Kaggle
**Dataset Link:** https://www.kaggle.com/datasets/rohanrao/formula-1-world-championship-1950-2020
**Input folder:** `raw_data/` (the files exactly as downloaded from Kaggle)
**Output folder:** `df/` (the preprocessed files used for the upload)

## Objective

1. Reading the raw Kaggle CSV files
2. Keeping only the 12 tables and the columns used in our database design
3. Formatting the date columns
4. Formatting the fastest-lap time
5. Saving the preprocessed files into the `df/` folder
6. Checking the preprocessed files

## Team Members

1. 202618016 - Krishika Lalwani
2. 202618013 - Devanshi vora
3. 202618052 - Lakshita Pagaria

In [12]:
import os
import math
import shutil
import pandas as pd

## Set the input and output folders

Download the dataset from Kaggle, unzip it, and place all the CSV files in the `raw_data` folder.

In [13]:
RAW_DIR = "../data/raw"
OUT_DIR = "../data/preprocessed"

os.makedirs(OUT_DIR, exist_ok=True)
print("Files in raw_data:", sorted(f for f in os.listdir(RAW_DIR) if f.endswith(".csv")))

Files in raw_data: ['circuits.csv', 'constructor_results.csv', 'constructor_standings.csv', 'constructors.csv', 'driver_standings.csv', 'drivers.csv', 'lap_times.csv', 'pit_stops.csv', 'qualifying.csv', 'races.csv', 'results.csv', 'seasons.csv', 'sprint_results.csv', 'status.csv']


## Tables and columns used in the database

Our schema uses 12 of the 14 Kaggle files (`seasons` and `constructor_results` are not used).
For each table only the columns of our relational schema are kept; the remaining columns
(for example `url`, `lat`, `lng`, `number`, `positionText`, `time`, `fastestLapSpeed`) are removed.
The column order follows the original files.

In [14]:
table_columns = {
    "circuits": ["circuitId", "circuitRef", "name", "location", "country"],
    "constructors": ["constructorId", "constructorRef", "name", "nationality"],
    "drivers": ["driverId", "driverRef", "forename", "surname", "dob", "nationality"],
    "status": ["statusId", "status"],
    "races": ["raceId", "year", "round", "circuitId", "name", "date", "time"],
    "results": ["resultId", "raceId", "driverId", "constructorId", "grid", "position",
                "positionOrder", "points", "laps", "rank", "fastestLapTime", "statusId"],
    "sprint_results": ["resultId", "raceId", "driverId", "constructorId", "grid", "position",
                       "positionOrder", "points", "laps", "statusId"],
    "qualifying": ["qualifyId", "raceId", "driverId", "constructorId", "position"],
    "lap_times": ["raceId", "driverId", "lap", "milliseconds"],
    "pit_stops": ["raceId", "driverId", "stop", "lap", "milliseconds"],
    "driver_standings": ["driverStandingsId", "raceId", "driverId", "points", "position", "wins"],
    "constructor_standings": ["constructorStandingsId", "raceId", "constructorId", "points", "position", "wins"]
}

## Read the raw files

Every value is read as text so that numbers, IDs and the `\N` missing-value marker are kept exactly as they are
in the original files.

In [15]:
raw = {}
for table_name in table_columns:
    raw[table_name] = pd.read_csv(f"{RAW_DIR}/{table_name}.csv", dtype=str, keep_default_na=False)
    print(f"{table_name:<22} {raw[table_name].shape[0]:>7} rows  {raw[table_name].shape[1]:>2} columns")

circuits                    77 rows   5 columns
constructors               212 rows   4 columns
drivers                    861 rows   6 columns
status                     139 rows   2 columns
races                     1125 rows  18 columns
results                  26759 rows  18 columns
sprint_results             360 rows  16 columns
qualifying               10494 rows   9 columns
lap_times               589081 rows   6 columns
pit_stops                11371 rows   7 columns
driver_standings         34863 rows   7 columns
constructor_standings    13391 rows   7 columns


## Keep the required columns

In [16]:
data = {}
for table_name, columns in table_columns.items():
    data[table_name] = raw[table_name][columns].copy()
    removed = [c for c in raw[table_name].columns if c not in columns]
    print(f"{table_name:<22} removed: {removed}")

circuits               removed: []
constructors           removed: []
drivers                removed: []
status                 removed: []
races                  removed: ['url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time']
results                removed: ['number', 'positionText', 'time', 'milliseconds', 'fastestLap', 'fastestLapSpeed']
sprint_results         removed: ['number', 'positionText', 'time', 'milliseconds', 'fastestLap', 'fastestLapTime']
qualifying             removed: ['number', 'q1', 'q2', 'q3']
lap_times              removed: ['position', 'time']
pit_stops              removed: ['time', 'duration']
driver_standings       removed: ['positionText']
constructor_standings  removed: ['positionText']


## Format the dates

`drivers.dob` and `races.date` are written as **DD-MM-YYYY** (e.g. `1985-01-07` becomes `07-01-1985`).
Dates before 1900 (the birth dates of seven early drivers) are kept in their original `YYYY-MM-DD` form.

In [17]:
def format_date(value):
    if value in ("\\N", ""):
        return value
    year, month, day = value.split("-")
    if int(year) < 1900:
        return value
    return f"{day}-{month}-{year}"

data["drivers"]["dob"] = data["drivers"]["dob"].map(format_date)
data["races"]["date"] = data["races"]["date"].map(format_date)

print(data["drivers"][["driverRef", "dob"]].head())
print(data["races"][["name", "date"]].head())

    driverRef         dob
0    hamilton  07-01-1985
1    heidfeld  10-05-1977
2     rosberg  27-06-1985
3      alonso  29-07-1981
4  kovalainen  19-10-1981
                    name        date
0  Australian Grand Prix  29-03-2009
1   Malaysian Grand Prix  05-04-2009
2     Chinese Grand Prix  19-04-2009
3     Bahrain Grand Prix  26-04-2009
4     Spanish Grand Prix  10-05-2009


## Format the fastest-lap time

`results.fastestLapTime` is written as **MM:SS.s**, rounded to one tenth of a second
(e.g. `1:27.452` becomes `01:27.5`). Missing values (`\N`) are kept.

In [18]:
def format_lap_time(value):
    if value in ("\\N", ""):
        return value
    minutes, seconds = value.split(":")
    tenths = math.floor((int(minutes) * 60 + float(seconds)) * 10 + 0.5 + 1e-9)
    mm, rest = divmod(tenths, 600)
    return f"{mm:02d}:{rest // 10:02d}.{rest % 10}"

data["results"]["fastestLapTime"] = data["results"]["fastestLapTime"].map(format_lap_time)
print(data["results"][["resultId", "fastestLapTime"]].head())

  resultId fastestLapTime
0        1        01:27.5
1        2        01:27.7
2        3        01:28.1
3        4        01:28.6
4        5        01:27.4


## Save the preprocessed files

The files are saved as UTF-8 CSV files in the `df` folder. `status.csv` needs no changes,
so it is copied as it is.

In [19]:
for table_name, df in data.items():
    if table_name == "status":
        shutil.copyfile(f"{RAW_DIR}/status.csv", f"{OUT_DIR}/status.csv")
    else:
        df.to_csv(f"{OUT_DIR}/{table_name}.csv", index=False, encoding="utf-8", lineterminator="\r\n")
    print(f"Saved {OUT_DIR}/{table_name}.csv")

Saved ../data/preprocessed/circuits.csv
Saved ../data/preprocessed/constructors.csv
Saved ../data/preprocessed/drivers.csv
Saved ../data/preprocessed/status.csv
Saved ../data/preprocessed/races.csv
Saved ../data/preprocessed/results.csv
Saved ../data/preprocessed/sprint_results.csv
Saved ../data/preprocessed/qualifying.csv
Saved ../data/preprocessed/lap_times.csv
Saved ../data/preprocessed/pit_stops.csv
Saved ../data/preprocessed/driver_standings.csv
Saved ../data/preprocessed/constructor_standings.csv


## Check the preprocessed files

In [20]:
summary = []
for table_name in table_columns:
    df = pd.read_csv(f"{OUT_DIR}/{table_name}.csv", dtype=str, keep_default_na=False)
    summary.append({
        "table": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": ", ".join(df.columns)
    })

summary = pd.DataFrame(summary)
pd.set_option("display.max_colwidth", 120)
print(summary.to_string(index=False))

                table   rows  columns                                                                                                           column_names
             circuits     77        5                                                                         circuitId, circuitRef, name, location, country
         constructors    212        4                                                                       constructorId, constructorRef, name, nationality
              drivers    861        6                                                               driverId, driverRef, forename, surname, dob, nationality
               status    139        2                                                                                                       statusId, status
                races   1125        7                                                                       raceId, year, round, circuitId, name, date, time
              results  26759       12 resultId, raceId, dr